# Setup

In [ ]:
%run common.py
import sys
sys.path.append("../../legal-data-clustering/")
%run '../../legal-data-clustering/legal_data_clustering/utils/graph_api.py'

In [ ]:
pd.options.display.float_format = '{:,.0f}'.format

def sortable_citekey(text):
    abk, nr = text.split('_')
    match = re.fullmatch('\d+([a-z]*)', nr, flags=re.IGNORECASE)
    extra_len = len(match[1])
    return abk + '_' + nr.zfill(4 + extra_len)

In [ ]:
de_crossreference_path = f'../../legal-networks-data/de/4_crossreference_graph/seqitems'

In [ ]:
G_crossreference = nx.read_gpickle(f'{de_crossreference_path}/2019-01-01.gpickle.gz')

In [ ]:
counter_art = defaultdict(int)
counter_para= defaultdict(int)

for node, heading in nx.get_node_attributes(G_crossreference, 'heading').items():
    if  G_crossreference.nodes[node]['type'] == 'seqitem':
        abk = node.split('_')[1]
        if heading.startswith('§'):
            counter_para[abk] += 1
        elif heading.lower().startswith('art'):
            counter_art[abk] += 1
        
abk_units = {
    abk: ('§' if counter_para[abk] > counter_art[abk] else 'Art')
    for abk in (set(counter_art) | set(counter_para))
}

# Load

In [ ]:
G = nx.read_gpickle('../../legal-networks-data/de_decisions/2_network.gpickle.gz')

In [ ]:
propagate_attrs_to_descendents(G, ['gericht', 'spruchkoerper', 'datum'])
nx.set_node_attributes(G, {n: n.split('_')[0] for n in G.nodes}, 'decision_key')

In [ ]:
H = make_occurrence_graph(
    G, 
    decision_level='seqitem', 
    decision_to_edge_attrs=['gericht', 'spruchkoerper', 'datum', 'decision_key']
)

In [ ]:
nrs = sorted([c for c in H.nodes if c.startswith('GG_') and sortable_citekey(c) < 'GG_0020' ], key=sortable_citekey)

In [ ]:
X = nx.MultiGraph()
X.add_nodes_from(H.nodes)
X.add_edges_from([
    (u, v, (d['gericht'], d['datum'], d['spruchkoerper']), d)
    for u, v, k, d in H.edges(keys=True, data=True)
])
print(len(H.edges), len(X.edges))
H = X

In [ ]:
H1 = H.edge_subgraph((u,v,k) for u,v,k,datum in H.edges(keys=True, data='datum') if datum < '2015-01-01')
H2 = H.edge_subgraph((u,v,k) for u,v,k,datum in H.edges(keys=True, data='datum') if datum > '2015-01-01')

In [ ]:
def fix_pandas_to_latex_no_index(tex):
    return re.sub(
        r'\\toprule\n([^&]+)\s&\s\\multicolumn\{(\d)\}\{(\w)\}', 
        '\\\\toprule\n\\\\multicolumn{\g<2>}{\g<3>}{\g<1>} & \\\\multicolumn{\g<2>}{\g<3>}', tex)

In [ ]:
dfs = []
for nr in nrs:
    counts = Counter([v for u, v in H.edges(nr)])
    art_cnts = [
        (n, cnt) 
        for n, cnt in counts.most_common() 
        if not n.startswith('GG_')
    ][:50]
    art_cnts_formatted = select_citekey(art_cnts, abk_units)
    df = pd.DataFrame(
        [(cnt, n) for n, cnt in art_cnts_formatted],
        columns=['#', 'Norm']
    )
    dfs.append(df)
df = pd.concat(dfs, keys=['Art. ' + ' '.join(reversed(nr.split('_'))) for nr in nrs], axis=1)
df = df.drop(['Art. 6a GG', 'Art. 15 GG', 'Art. 17a GG'], axis=1)

In [ ]:
def create_tex_tables(df, table_indices, path):
    cols_per_row = 10
    tex = ''
    for table_idx in table_indices:
        df_part = df.iloc[:, table_idx*cols_per_row : table_idx*cols_per_row+cols_per_row ]
        tex += fix_pandas_to_latex_no_index(df_part.to_latex(index=False))
        tex += '\n\\vspace{3mm}\n'
    tex = tex[:-len('\vspace{3mm}\n')]
    tex = re.sub(r"-\d{4}", "...", tex)
    with open(path, 'w') as f:
        f.write(tex)

assert len(df.columns) == 40
df = df.head(26).fillna('')
create_tex_tables(df, [0,1,], '../tables/mikro_gg_fix_1.tex')
create_tex_tables(df, [2,3], '../tables/mikro_gg_fix_2.tex')

# Diff 2010-2014 vs 2016-2020

In [ ]:
dfs = []
for nr in nrs:
    counts = Counter([v for u, v in H2.edges(nr)])
    counts.subtract(Counter([v for u, v in H1.edges(nr)]))
    art_cnts = [
        (n, cnt) 
        for n, cnt in counts.most_common() 
        if not n.startswith('GG_')
    ]
    if len(art_cnts) > 26:
        art_cnts = art_cnts[:13] + art_cnts[-13:] 
    art_cnts_formatted = select_citekey(art_cnts, abk_units)
    df = pd.DataFrame(
        [(cnt, n) for n, cnt in art_cnts_formatted],
        columns=['#', 'Norm']
    )
    dfs.append(df)
df = pd.concat(dfs, keys=['Art. ' + ' '.join(reversed(nr.split('_'))) for nr in nrs], axis=1)
df = df.drop(['Art. 6a GG', 'Art. 15 GG', 'Art. 17a GG'], axis=1)

assert len(df.columns) == 40
df = df.fillna('')
create_tex_tables(df, [0,1], '../tables/mikro_gg_diff_1.tex')
create_tex_tables(df, [2,3], '../tables/mikro_gg_diff_2.tex')

# Instant inspection

In [ ]:
l = sorted([d for u,v,k,d in H.edges('GG_12', keys=True, data=True) if v == 'BetrVG_75'], key=lambda x: x['datum'])
print(len(l))
l